# Inference with a trained metric-matching model

Metric matching trains a network that outputs a low-rank factor $U(x, h)$ of
the carré du champ (CDC) matrix $\Gamma(x, h) = U^\top U$. This matrix
converges to the projection onto the tangent space of the data manifold at
$x$ (Corollary 4.2 of the paper), so a single forward pass gives you, at any
point, the tangent directions (the top eigenvectors of $G = U^\top U$), the
local intrinsic dimension (from how fast the eigenvalues decay), and a
Riemannian metric you can use downstream, for intrinsic gradients or
on-manifold optimization.

This notebook loads a trained MNIST metric model and goes through the
analysis behind Figure 3 of the paper: eigenvector visualizations, tangent
perturbations, eigenvalue spectra, and what the bandwidth $h$ does.

You need `pip install -e .` from the repo root and a trained MNIST metric
model. The paper's MNIST model is on the
[models-v1 release page](https://github.com/jacobbamb/metric-matching/releases/tag/models-v1).
Download it into `models/` at the repo root (72 MB):

```bash
mkdir -p models
curl -L https://github.com/jacobbamb/metric-matching/releases/download/models-v1/mnist-metric.tar.gz | tar xz -C models
```

This gives `models/mnist/metric_15-44-39/` with the run config and the
epoch-1999 checkpoint, the one behind the paper's MNIST figures. Or train your
own with the two-step recipe from the README:

```bash
python scripts/train_score.py --config-name mnist   # step 1: score model
python scripts/train.py --config-name mnist         # step 2: metric model
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Resolve the repo root whether this runs from notebooks/ or the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Where to load the metric model from. The released paper checkpoint (see the
# download step above) is used if present; otherwise we fall back to your own
# training runs.
MODEL_ROOT = REPO_ROOT / "models/mnist"
EPOCH = 1999  # the epoch used in the paper
if not any(MODEL_ROOT.rglob("*.ckpt")):
    MODEL_ROOT = REPO_ROOT / "outputs/metric/mnist"
    EPOCH = None  # newest last.ckpt of your own runs
    if not any(MODEL_ROOT.rglob("*.ckpt")):
        raise FileNotFoundError(
            "No MNIST metric checkpoint found. Download the released model into "
            f"{REPO_ROOT / 'models'} (see above) or train one with scripts/train.py."
        )

H = 7.0  # bandwidth used for the paper's MNIST figures
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"model root: {MODEL_ROOT}\ndevice: {DEVICE}")


## Load the model

`load_metric_model_from_checkpoint` finds the checkpoint (an exact epoch, or
the newest `last.ckpt`), rebuilds the network from the config saved with that
run, and prints which weights it picked, so you always know what you are
looking at.

In [ ]:
from metric_matching.systems.mm_system import load_metric_model_from_checkpoint

model = load_metric_model_from_checkpoint(epoch=EPOCH, preferred_root=str(MODEL_ROOT))
model = model.to(DEVICE).eval()
rank, D = model.rank, model.out_dim
print(f"rank r = {rank}, ambient dimension D = {D}")

## Load validation images

We go through the same `MNISTDataModule` as training, so the normalization is
exactly what the model saw.

In [ ]:
from metric_matching.data.datamodule import MNISTDataModule

dm = MNISTDataModule(data_dir=str(REPO_ROOT / "data"), batch_size=256, noise_level=0.0)
dm.prepare_data()
dm.setup()
val = dm.mnist_val

# MNIST normalization constants, for turning tensors back into images.
MEAN, STD = 0.1307, 0.3081
denorm = lambda t: (t * STD + MEAN).clamp(0, 1)

indices = [0, 1, 2, 4]  # a 7, a 2, a 1 and a 4
imgs = torch.stack([val[i][0] for i in indices])
labels = [int(val[i][1]) for i in indices]
print("digits:", labels)

## Compute the CDC factor and its eigendecomposition

The model maps a batch of flattened images and a bandwidth $h$ to
$U \in \mathbb{R}^{B \times r \times D}$. We never form the $D \times D$
matrix $G = U^\top U$. Instead `batched_eig_of_UtU` gets its eigenvalues and
eigenvectors from a thin SVD of $U$, in descending order.

In [ ]:
from metric_matching.utils.spectra import batched_eig_of_UtU

x = imgs.reshape(len(indices), -1).to(DEVICE)
h = torch.full((x.shape[0],), H, device=DEVICE)
with torch.no_grad():
    U = model(h, x)  # [B, r, D]

evals, evecs = batched_eig_of_UtU(U.float().cpu())  # [B, r], [B, D, r]
print("U:", tuple(U.shape), " evals:", tuple(evals.shape), " evecs:", tuple(evecs.shape))

## Tangent directions (paper Fig. 3a)

The leading eigenvectors are interpretable directions in image space. The
first few tend to redraw or thicken the digit and shift its strokes.
High-index ones (the 50th and 100th here) look like noise, because they point
off the data manifold.

In [ ]:
EVEC_COLS = [0, 1, 2, 3, 4, 49, 99]  # v1..v5, v50, v100

fig, axes = plt.subplots(
    len(indices), 1 + len(EVEC_COLS), figsize=(1.35 * (1 + len(EVEC_COLS)), 1.45 * len(indices))
)
for row, (img, lab) in enumerate(zip(imgs, labels)):
    axes[row, 0].imshow(denorm(img[0]), cmap="gray")
    axes[row, 0].set_ylabel(f"digit {lab}", fontsize=9)
    for col, k in enumerate(EVEC_COLS, start=1):
        v = evecs[row, :, k].reshape(28, 28)
        m = v.abs().max()
        axes[row, col].imshow(v, cmap="bwr", vmin=-m, vmax=m)
        if row == 0:
            axes[row, col].set_title(f"$v_{{{k + 1}}}$", fontsize=10)
for ax in axes.flat:
    ax.set_xticks([]), ax.set_yticks([])
axes[0, 0].set_title("image", fontsize=10)
fig.suptitle("Eigenvectors of the learned CDC matrix", y=1.0)
plt.tight_layout()
plt.show()

## Eigenvalue spectra and local intrinsic dimension (paper Fig. 3b)

The eigenvalues of $G$ measure how much local variability each tangent
direction carries. They decay sharply, which reflects the low intrinsic
dimension of MNIST, and the cumulative explained local variance differs
between classes. The digit 1, visually the simplest, saturates fastest.

In [ ]:
N_PER_CLASS = 16
BATCH = 64

# Collect N_PER_CLASS validation images of each digit.
by_class = {c: [] for c in range(10)}
for img, lab in val:
    lab = int(lab)
    if len(by_class[lab]) < N_PER_CLASS:
        by_class[lab].append(img.reshape(-1))
    if all(len(v) == N_PER_CLASS for v in by_class.values()):
        break
xs = torch.cat([torch.stack(by_class[c]) for c in range(10)])
ys = torch.arange(10).repeat_interleave(N_PER_CLASS)
print(len(xs))

all_evals = []
with torch.no_grad():
    for i in range(0, len(xs), BATCH):
        xb = xs[i : i + BATCH].to(DEVICE)
        hb = torch.full((len(xb),), H, device=DEVICE)
        # U U^T has the same nonzero eigenvalues as U^T U, but is only r x r.
        # Keep computation on DEVICE and skip eigenvectors for these plots.
        U = model(hb, xb).float()
        gram = U @ U.transpose(-2, -1)
        ev = torch.linalg.eigvalsh(gram).flip(-1).clamp_min(0)
        all_evals.append(ev.cpu())
        print(f"Processed {min(i + BATCH, len(xs))}/{len(xs)} images", flush=True)
all_evals = torch.cat(all_evals)  # [160, r]

cum = all_evals.cumsum(dim=1) / all_evals.sum(dim=1, keepdim=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
ax1.semilogy(all_evals.mean(0), marker=".", ms=4)
ax1.set_xlabel("eigenvalue index")
ax1.set_ylabel("eigenvalue")
ax1.set_title("Mean spectrum (log scale)")
for c in range(10):
    ax2.plot(range(1, cum.shape[1] + 1), cum[ys == c].mean(0), label=str(c), lw=1.4)
ax2.set_xlabel("eigenvalue index")
ax2.set_ylabel("explained local variance")
ax2.set_title("Class-wise cumulative eigenvalues")
ax2.legend(ncol=2, fontsize=8, title="digit")
plt.tight_layout()
plt.show()

## Where to go next

- To train your own models, follow the two-step recipe in the README. It
  covers all five datasets with the paper's settings.
- With $G$ in hand, the intrinsic gradient of any function $f$ is the
  matrix-vector product $G(x)\,\partial f(x)$ (Sec. 4.3 and 4.4 of the
  paper). From `U` that is `U.transpose(1, 2) @ (U @ grad_f)`, again without
  forming $G$.
- The eigenvalue decay shown above is what the local intrinsic-dimension
  estimate in Sec. 5 is built on.